In [ ]:
#STEP 0: Mounting Drive + quick check
from google.colab import drive
drive.mount('/content/drive')

!ls "/content/drive/MyDrive/CAPSTONE PROJECT"

In [ ]:
#STEP 1: Imports

import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#sklearn tools
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize
from sklearn.inspection import permutation_importance

In [ ]:
#xgboost
from xgboost import XGBClassifier

In [ ]:
#STEP 2: Important Settings + Results Folder

SEED = 42
TARGET_COL = "label"

#results folder — all plots + CSVs will save here
RESULTS_DIR = "/content/drive/MyDrive/CAPSTONE PROJECT/XG boost"
os.makedirs(RESULTS_DIR, exist_ok=True)

#reproducibility
np.random.seed(SEED)
random.seed(SEED)

print("Setup done.")
print("Results will save to:", RESULTS_DIR)

In [ ]:
#STEP 3: Load the CSV + sanity check

CSV_PATH = "/content/drive/MyDrive/CAPSTONE PROJECT/9 CLASSES DATA SET/SC_Dataset_9_Classes.csv"

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nLabel counts:")
print(df[TARGET_COL].value_counts())

print("\nAny nulls?", df.isnull().sum().sum())

display(df.head(3))

In [ ]:
#STEP 4: Encode labels (text to numbers)

#mapping each class name to a number
label_map = {
    "pigmented benign keratosis" : 0,
    "melanoma"                   : 1,
    "basal cell carcinoma"       : 2,
    "nevus"                      : 3,
    "squamous cell carcinoma"    : 4,
    "vascular lesion"            : 5,
    "actinic keratosis"          : 6,
    "dermatofibroma"             : 7,
    "seborrheic keratosis"       : 8
}

#apply the mapping
df[TARGET_COL] = df[TARGET_COL].map(label_map).astype(int)

#class names in order (we'll use this for plots later)
CLASS_NAMES = list(label_map.keys())

print("Label encoding done.")
print("\nEncoded label counts:")
print(df[TARGET_COL].value_counts().sort_index())
print("\nClass names:", CLASS_NAMES)

In [ ]:
#STEP 5: Stratified 80/20 Train/Test Split

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,          #this ensures all 9 classes are represented in both train and test
    random_state=SEED
)

print("Train size:", X_train.shape)
print("Test size :", X_test.shape)

print("\nTrain label counts:")
print(y_train.value_counts().sort_index())

print("\nTest label counts:")
print(y_test.value_counts().sort_index())

In [ ]:
#STEP 6A: Channel Ratio Features
#dividing channels to capture color dominance between R, G, B

X_train["ratio_rg"] = X_train["avg_r"] / (X_train["avg_g"] + 1e-6)
X_train["ratio_rb"] = X_train["avg_r"] / (X_train["avg_b"] + 1e-6)
X_train["ratio_gb"] = X_train["avg_g"] / (X_train["avg_b"] + 1e-6)

X_test["ratio_rg"] = X_test["avg_r"] / (X_test["avg_g"] + 1e-6)
X_test["ratio_rb"] = X_test["avg_r"] / (X_test["avg_b"] + 1e-6)
X_test["ratio_gb"] = X_test["avg_g"] / (X_test["avg_b"] + 1e-6)

print("Channel ratio features added.")
print("New shape:", X_train.shape)

In [ ]:
#STEP 6B: Color Contrast Features
#subtracting channels to capture how different the colors are from each other

X_train["contrast_rg"] = X_train["avg_r"] - X_train["avg_g"]
X_train["contrast_rb"] = X_train["avg_r"] - X_train["avg_b"]
X_train["contrast_gb"] = X_train["avg_g"] - X_train["avg_b"]

X_test["contrast_rg"] = X_test["avg_r"] - X_test["avg_g"]
X_test["contrast_rb"] = X_test["avg_r"] - X_test["avg_b"]
X_test["contrast_gb"] = X_test["avg_g"] - X_test["avg_b"]

print("Color contrast features added.")
print("New shape:", X_train.shape)

In [ ]:
#STEP 6C: Color Moment Features
#overall brightness, color range, and color variation across R, G, B

X_train["brightness"]  = (X_train["avg_r"] + X_train["avg_g"] + X_train["avg_b"]) / 3
X_train["color_range"] = X_train[["avg_r","avg_g","avg_b"]].max(axis=1) - X_train[["avg_r","avg_g","avg_b"]].min(axis=1)
X_train["color_std"]   = X_train[["avg_r","avg_g","avg_b"]].std(axis=1)

X_test["brightness"]  = (X_test["avg_r"] + X_test["avg_g"] + X_test["avg_b"]) / 3
X_test["color_range"] = X_test[["avg_r","avg_g","avg_b"]].max(axis=1) - X_test[["avg_r","avg_g","avg_b"]].min(axis=1)
X_test["color_std"]   = X_test[["avg_r","avg_g","avg_b"]].std(axis=1)

print("Color moment features added.")
print("New shape:", X_train.shape)

In [ ]:
#STEP 6D: ABCD Shape Features
#asymmetry, border irregularity, and diameter estimate from existing shape columns

X_train["asymmetry_ratio"]     = X_train["min_area_rect_width"] / (X_train["min_area_rect_height"] + 1e-6)
X_train["border_irregularity"] = X_train["min_enc_circle_area"] / (X_train["min_area_rect_width"] * X_train["min_area_rect_height"] + 1e-6)
X_train["diameter_estimate"]   = (X_train["min_area_rect_width"] + X_train["min_area_rect_height"]) / 2

X_test["asymmetry_ratio"]     = X_test["min_area_rect_width"] / (X_test["min_area_rect_height"] + 1e-6)
X_test["border_irregularity"] = X_test["min_enc_circle_area"] / (X_test["min_area_rect_width"] * X_test["min_area_rect_height"] + 1e-6)
X_test["diameter_estimate"]   = (X_test["min_area_rect_width"] + X_test["min_area_rect_height"]) / 2

print("ABCD shape features added.")
print("New shape:", X_train.shape)

In [ ]:
#STEP 6E: Summary of all new engineered features

new_features = [
    "ratio_rg", "ratio_rb", "ratio_gb",
    "contrast_rg", "contrast_rb", "contrast_gb",
    "brightness", "color_range", "color_std",
    "asymmetry_ratio", "border_irregularity",
    "diameter_estimate"
]

print("Total new features added:", len(new_features))
print("Total feature count now:", X_train.shape[1])
print("\nNew features:", new_features)

In [ ]:
#STEP 7: Applying SMOTE on training data only

from imblearn.over_sampling import SMOTE

print("Before SMOTE:")
print(y_train.value_counts().sort_index())

smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train_sm).value_counts().sort_index())

print("\nX_train shape after SMOTE:", X_train_sm.shape)

In [ ]:
#STEP 8:XGBoost Pipeline

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=9,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    ))
])

print("Pipeline ready.")
print(pipe)

In [ ]:
#STEP 9: Hyperparameter Tuning with RandomizedSearchCV

param_dist = {
    "xgb__n_estimators":     [100, 200, 300, 400],
    "xgb__max_depth":        [3, 4, 5],
    "xgb__learning_rate":    [0.03, 0.05, 0.1],
    "xgb__subsample":        [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0],
    "xgb__reg_lambda":       [1.0, 1.5],
    "xgb__reg_alpha":        [0.0, 0.1],
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=40,
    scoring="f1_macro",
    n_jobs=-1,
    cv=cv,
    verbose=1,
    random_state=SEED
)

#training on SMOTE'd training data
search.fit(X_train_sm, y_train_sm)

print("\nBEST PARAMETERS:")
print(json.dumps(search.best_params_, indent=2))

print("\nBest CV F1 macro:", round(search.best_score_, 4))

best_model = search.best_estimator_

In [ ]:
#STEP 10: Evaluating on test set

y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 4))

print("\nClassification Report (9-class XGBoost):")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#Saving trained model

import joblib

joblib.dump(best_model, "/content/drive/MyDrive/CAPSTONE PROJECT/XG boost/xgb_best_model.pkl")
print("Model saved to Drive!")

In [ ]:
#STEP 11: Confusion Matrix

fig, ax = plt.subplots(figsize=(12, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred, labels=list(range(9))),
    display_labels=CLASS_NAMES
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=True
)

ax.set_title("Confusion Matrix — XGBoost (9-class)", fontsize=16, pad=12)
ax.set_xlabel("Predicted label", fontsize=13)
ax.set_ylabel("True label", fontsize=13)

plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=10)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix_9class.png"), dpi=150)
plt.show()

print("Saved confusion matrix.")

In [ ]:
#STEP 12: ROC-AUC Curves

y_test_bin = label_binarize(y_test, classes=list(range(9)))

auc_macro = roc_auc_score(
    y_test_bin,
    y_proba,
    average="macro",
    multi_class="ovr"
)

print("ROC-AUC macro (OvR):", round(auc_macro, 4))

plt.figure(figsize=(10, 7))

for i, cname in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    plt.plot(fpr, tpr, label=cname)

plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.xlabel("False Positive Rate", fontsize=13)
plt.ylabel("True Positive Rate", fontsize=13)
plt.title(f"ROC Curves — XGBoost (9-class) | macro AUC = {auc_macro:.3f}", fontsize=15)
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "roc_curves_9class.png"), dpi=150)
plt.show()

print("Saved ROC curves.")

In [ ]:
#STEP 13: Feature Importance by Gain (top 30)

xgb_model    = best_model.named_steps["xgb"]
feature_names = X_train_sm.columns.tolist()

booster     = xgb_model.get_booster()
gain_scores = booster.get_score(importance_type="gain")

#map f0, f1... back to real feature names
mapped_scores = {}
for k, v in gain_scores.items():
    if k.startswith("f"):
        idx = int(k[1:])
        if idx < len(feature_names):
            mapped_scores[feature_names[idx]] = v

imp_df = (
    pd.DataFrame({
        "feature": list(mapped_scores.keys()),
        "gain"   : list(mapped_scores.values())
    })
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 features (gain):")
display(imp_df.head(20))

#plotting top 30
top_30 = imp_df.head(30).iloc[::-1]

plt.figure(figsize=(10, 8))
plt.barh(top_30["feature"], top_30["gain"], color="#1f77b4")
plt.xlabel("Gain", fontsize=13)
plt.ylabel("Feature", fontsize=13)
plt.title("Top 30 Feature Importances — XGBoost (9-class)", fontsize=15, pad=12)
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.xticks(fontsize=11)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "feature_importance_gain_9class.png"), dpi=150)
plt.show()

print("Saved feature importance plot.")

In [ ]:
#STEP 14: Permutation Importance

PI_DIR = os.path.join(RESULTS_DIR, "permutation_importance")
os.makedirs(PI_DIR, exist_ok=True)

pi = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1
)

pi_df = pd.DataFrame({
    "feature": X_test.columns,
    "pi_mean": pi.importances_mean,
    "pi_std" : pi.importances_std
}).sort_values("pi_mean", ascending=False).reset_index(drop=True)

#save full ranking
pi_df.to_csv(os.path.join(PI_DIR, "permutation_importance_full.csv"), index=False)

print("Top 10 features by permutation importance:")
display(pi_df.head(10))

#plot top 30
top_30 = pi_df.head(30).iloc[::-1]

plt.figure(figsize=(10, 8))
plt.barh(top_30["feature"], top_30["pi_mean"], xerr=top_30["pi_std"])
plt.xlabel("Mean decrease in F1 macro after shuffling", fontsize=12)
plt.title("Top 30 Permutation Importances — XGBoost (9-class)", fontsize=15)
plt.grid(axis="x", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(os.path.join(PI_DIR, "permutation_importance_top30.png"), dpi=150)
plt.show()

print("Saved permutation importance plot.")

In [ ]:
#STEP 15: SHAP Analysis

import shap

SHAP_DIR = os.path.join(RESULTS_DIR, "shap")
os.makedirs(SHAP_DIR, exist_ok=True)

shap_model = best_model.named_steps["xgb"]

explainer = shap.TreeExplainer(shap_model)

try:
    shap_exp  = explainer(X_test)
    shap_vals = shap_exp.values
except:
    shap_vals = explainer.shap_values(X_test)

#normalize to (n_samples, n_features, n_classes)
if isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
    shap_list = [shap_vals[:, :, i] for i in range(9)]
elif isinstance(shap_vals, list):
    shap_list = shap_vals

print("SHAP done.")
print("Shape per class:", shap_list[0].shape)

#summary plot per class
for i, cname in enumerate(CLASS_NAMES):
    plt.figure()
    shap.summary_plot(shap_list[i], X_test, show=False)
    plt.title(f"SHAP Summary — {cname}")
    out_path = os.path.join(SHAP_DIR, f"shap_summary_{i}_{cname.replace(' ','_')}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", out_path)

#global importance
mean_abs_shap = np.mean([np.abs(sv) for sv in shap_list], axis=(0, 1))

shap_global_df = pd.DataFrame({
    "feature"      : X_test.columns,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

shap_global_df.to_csv(os.path.join(SHAP_DIR, "shap_global_importance.csv"), index=False)

print("\nTop 10 global SHAP features:")
display(shap_global_df.head(10))

#global bar plot
top_30 = shap_global_df.head(30).iloc[::-1]

plt.figure(figsize=(10, 8))
plt.barh(top_30["feature"], top_30["mean_abs_shap"])
plt.xlabel("Mean |SHAP value|", fontsize=13)
plt.title("Top 30 Global SHAP Importances — XGBoost (9-class)", fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, "shap_global_top30.png"), dpi=150)
plt.show()

print("Saved global SHAP plot.")

In [ ]:
#STEP 16: Save Predictions + Error Analysis

results_df = X_test.copy()
results_df["true_label"] = y_test.values
results_df["pred_label"] = y_pred

#add probabilities for each class
prob_cols = [f"prob_{cname.replace(' ','_')}" for cname in CLASS_NAMES]
results_df[prob_cols] = y_proba

#save
save_path = os.path.join(RESULTS_DIR, "xgb_9class_predictions.csv")
results_df.to_csv(save_path, index=False)
print("Saved predictions to:", save_path)

#correct vs incorrect
results_df["correct"] = results_df["true_label"] == results_df["pred_label"]

print("\nCorrect vs incorrect:")
print(results_df["correct"].value_counts())

#where is the model getting confused
confusion_breakdown = (
    results_df
    .loc[~results_df["correct"], ["true_label", "pred_label"]]
    .value_counts()
    .reset_index(name="count")
    .head(15)
)

print("\nTop 15 misclassifications:")
display(confusion_breakdown)

In [ ]:
#STEP 17: SMOTE + Undersampling (combined approach)
#instead of fully balancing everything to 382
#we oversample minority classes halfway up and undersample majority classes halfway down
#this is less aggressive than full SMOTE

from imblearn.combine import SMOTETomek

print("Before SMOTETomek:")
print(y_train.value_counts().sort_index())

smote_tomek = SMOTETomek(random_state=SEED)
X_train_st, y_train_st = smote_tomek.fit_resample(X_train, y_train)

print("\nAfter SMOTETomek:")
print(pd.Series(y_train_st).value_counts().sort_index())

print("\nNew training shape:", X_train_st.shape)

In [ ]:
#STEP 18: Train XGBoost on SMOTETomek balanced data
pipe_st = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=9,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    ))
])

search_st = RandomizedSearchCV(
    estimator=pipe_st,
    param_distributions=param_dist,
    n_iter=40,
    scoring="f1_macro",
    n_jobs=-1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=1,
    random_state=SEED
)

search_st.fit(X_train_st, y_train_st)

print("\nBEST PARAMETERS (SMOTETomek):")
print(json.dumps(search_st.best_params_, indent=2))
print("\nBest CV F1 macro:", round(search_st.best_score_, 4))

best_model_st = search_st.best_estimator_

#evaluate on test set
y_pred_st = best_model_st.predict(X_test)

print("\nTest Accuracy (SMOTETomek):", round(accuracy_score(y_test, y_pred_st), 4))
print("\nClassification Report (SMOTETomek):")
print(classification_report(y_test, y_pred_st, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#STEP 19: XGBoost with class weights (no SMOTE at all)
#instead of generating synthetic samples
#we tell xgboost to penalize mistakes on minority classes more

from sklearn.utils.class_weight import compute_sample_weight

#compute sample weights based on class frequency
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

pipe_cw = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=9,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    ))
])

search_cw = RandomizedSearchCV(
    estimator=pipe_cw,
    param_distributions=param_dist,
    n_iter=40,
    scoring="f1_macro",
    n_jobs=-1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=1,
    random_state=SEED
)

#pass sample weights during fit
search_cw.fit(X_train, y_train, xgb__sample_weight=sample_weights)

print("\nBEST PARAMETERS (class weights):")
print(json.dumps(search_cw.best_params_, indent=2))
print("\nBest CV F1 macro:", round(search_cw.best_score_, 4))

best_model_cw = search_cw.best_estimator_

#evaluate on test set
y_pred_cw = best_model_cw.predict(X_test)

print("\nTest Accuracy (class weights):", round(accuracy_score(y_test, y_pred_cw), 4))
print("\nClassification Report (class weights):")
print(classification_report(y_test, y_pred_cw, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#STEP 20: Summary comparison of all 3 balancing approaches

comparison_df = pd.DataFrame([
    {
        "Approach"        : "SMOTE",
        "Test Accuracy"   : 0.6038,
        "F1 macro"        : 0.50,
        "Seborrheic F1"   : 0.00,
        "Vascular F1"     : 0.76,
        "Melanoma F1"     : 0.66
    },
    {
        "Approach"        : "SMOTETomek",
        "Test Accuracy"   : 0.5975,
        "F1 macro"        : 0.50,
        "Seborrheic F1"   : 0.06,
        "Vascular F1"     : 0.74,
        "Melanoma F1"     : 0.68
    },
    {
        "Approach"        : "Class Weights",
        "Test Accuracy"   : 0.6038,
        "F1 macro"        : 0.51,
        "Seborrheic F1"   : 0.00,
        "Vascular F1"     : 0.79,
        "Melanoma F1"     : 0.67
    }
])

display(comparison_df)

#save
comparison_df.to_csv(os.path.join(RESULTS_DIR, "balancing_comparison.csv"), index=False)
print("Saved comparison table.")

In [ ]:
#STEP 22a: Improved XGBoost with expanded param grid + combined balancing

from imblearn.combine import SMOTETomek
from sklearn.utils.class_weight import compute_sample_weight

#expanded param grid
param_dist_v2 = {
    "xgb__n_estimators":      [300, 400, 500, 600],
    "xgb__max_depth":         [3, 4, 5, 6],
    "xgb__learning_rate":     [0.03, 0.05, 0.1],
    "xgb__subsample":         [0.7, 0.8, 1.0],
    "xgb__colsample_bytree":  [0.7, 0.8, 1.0],
    "xgb__reg_lambda":        [1.0, 1.5, 2.0],
    "xgb__reg_alpha":         [0.0, 0.1, 0.2],
    "xgb__min_child_weight":  [1, 3, 5],
    "xgb__gamma":             [0, 0.1, 0.2],
}

#SMOTETomek on training data
print("Applying SMOTETomek...")
smote_tomek = SMOTETomek(random_state=SEED)
X_train_v2, y_train_v2 = smote_tomek.fit_resample(X_train, y_train)

print("After SMOTETomek:")
print(pd.Series(y_train_v2).value_counts().sort_index())

#compute sample weights on top of SMOTETomek
sample_weights_v2 = compute_sample_weight(class_weight="balanced", y=y_train_v2)

pipe_v2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=9,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    ))
])

search_v2 = RandomizedSearchCV(
    estimator=pipe_v2,
    param_distributions=param_dist_v2,
    n_iter=80,
    scoring="f1_macro",
    n_jobs=-1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=1,
    random_state=SEED
)

print("\nTraining improved model — this will take ~15 mins...")
search_v2.fit(X_train_v2, y_train_v2, xgb__sample_weight=sample_weights_v2)

print("\nBEST PARAMETERS (v2):")
print(json.dumps(search_v2.best_params_, indent=2))
print("\nBest CV F1 macro:", round(search_v2.best_score_, 4))

best_model_v2 = search_v2.best_estimator_

#evaluate on test set
y_pred_v2  = be

In [ ]:
#STEP 22b:Improved XGBoost v2 (lightweight version)

from imblearn.combine import SMOTETomek
from sklearn.utils.class_weight import compute_sample_weight

#lightweight param grid
param_dist_v2 = {
    "xgb__n_estimators":     [200, 300, 400],
    "xgb__max_depth":        [4, 5],
    "xgb__learning_rate":    [0.05, 0.1],
    "xgb__subsample":        [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0],
    "xgb__min_child_weight": [1, 3],
    "xgb__gamma":            [0, 0.1],
}

#SMOTETomek
print("Applying SMOTETomek...")
smote_tomek = SMOTETomek(random_state=SEED)
X_train_v2, y_train_v2 = smote_tomek.fit_resample(X_train, y_train)

#class weights on top
sample_weights_v2 = compute_sample_weight(class_weight="balanced", y=y_train_v2)

pipe_v2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=9,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    ))
])

search_v2 = RandomizedSearchCV(
    estimator=pipe_v2,
    param_distributions=param_dist_v2,
    n_iter=25,              #nice and fast
    scoring="f1_macro",
    n_jobs=-1,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),  #3 folds instead of 5
    verbose=1,
    random_state=SEED
)

search_v2.fit(X_train_v2, y_train_v2, xgb__sample_weight=sample_weights_v2)

print("\nBEST PARAMETERS (v2):")
print(json.dumps(search_v2.best_params_, indent=2))
print("\nBest CV F1 macro:", round(search_v2.best_score_, 4))

best_model_v2 = search_v2.best_estimator_

y_pred_v2  = best_model_v2.predict(X_test)
y_proba_v2 = best_model_v2.predict_proba(X_test)

print("\nTest Accuracy (v2):", round(accuracy_score(y_test, y_pred_v2), 4))
print("\nClassification Report (v2):")
print(classification_report(y_test, y_pred_v2, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#STEP 22c: Improved XGBoost v2 (class weights only, no SMOTE)

from sklearn.utils.class_weight import compute_sample_weight

#compute weights on original training data
sample_weights_v2 = compute_sample_weight(class_weight="balanced", y=y_train)

pipe_v2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=9,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    ))
])

param_dist_v2 = {
    "xgb__n_estimators":     [200, 300, 400],
    "xgb__max_depth":        [4, 5],
    "xgb__learning_rate":    [0.05, 0.1],
    "xgb__subsample":        [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0],
    "xgb__min_child_weight": [1, 3],
    "xgb__gamma":            [0, 0.1],
}

search_v2 = RandomizedSearchCV(
    estimator=pipe_v2,
    param_distributions=param_dist_v2,
    n_iter=25,
    scoring="f1_macro",
    n_jobs=-1,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    verbose=1,
    random_state=SEED
)

search_v2.fit(X_train, y_train, xgb__sample_weight=sample_weights_v2)

print("\nBEST PARAMETERS (v2):")
print(json.dumps(search_v2.best_params_, indent=2))
print("\nBest CV F1 macro:", round(search_v2.best_score_, 4))

best_model_v2 = search_v2.best_estimator_

y_pred_v2 = best_model_v2.predict(X_test)

print("\nTest Accuracy (v2):", round(accuracy_score(y_test, y_pred_v2), 4))
print("\nClassification Report (v2):")
print(classification_report(y_test, y_pred_v2, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#STEP 23: Evaluate on balanced test subset (16 per class)

#combine X_test and y_test into one dataframe
test_df = X_test.copy()
test_df["label"] = y_test.values

#sample 16 per class
test_balanced = (
    test_df
    .groupby("label")
    .sample(n=16, random_state=SEED)
    .reset_index(drop=True)
)

#split back into X and y
X_test_bal = test_balanced.drop(columns=["label"])
y_test_bal  = test_balanced["label"]

print("Balanced test set shape:", X_test_bal.shape)
print("\nClass counts:")
print(y_test_bal.value_counts().sort_index())

#evaluate best model on balanced test set
y_pred_bal  = best_model.predict(X_test_bal)
y_proba_bal = best_model.predict_proba(X_test_bal)

print("\nTest Accuracy (balanced):", round(accuracy_score(y_test_bal, y_pred_bal), 4))
print("\nClassification Report (balanced 16 per class):")
print(classification_report(y_test_bal, y_pred_bal, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#STEP 24: Dropping seborrheic keratosis (class 8) and retrain
#only 64 training samples....not enough to learn from

#dropping from full dataset before splitting
df_8class = df[df[TARGET_COL] != 8].copy()

print("Original dataset size:", len(df))
print("8-class dataset size:", len(df_8class))
print("\nClass counts:")
print(df_8class[TARGET_COL].value_counts().sort_index())

In [ ]:
#STEP 25: Split + SMOTE + retrain on 8 classes

from imblearn.over_sampling import SMOTE

#split — features already engineered in df_8class
X_8 = df_8class.drop(columns=[TARGET_COL])
y_8 = df_8class[TARGET_COL].astype(int)

X_train_8, X_test_8, y_train_8, y_test_8 = train_test_split(
    X_8, y_8,
    test_size=0.20,
    stratify=y_8,
    random_state=SEED
)

#SMOTE on train only
smote = SMOTE(random_state=SEED)
X_train_8sm, y_train_8sm = smote.fit_resample(X_train_8, y_train_8)

print("After SMOTE:")
print(pd.Series(y_train_8sm).value_counts().sort_index())

#pipeline
pipe_8 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=8,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    ))
])

search_8 = RandomizedSearchCV(
    estimator=pipe_8,
    param_distributions=param_dist,
    n_iter=40,
    scoring="f1_macro",
    n_jobs=-1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=1,
    random_state=SEED
)

search_8.fit(X_train_8sm, y_train_8sm)

print("\nBest CV F1 macro:", round(search_8.best_score_, 4))

best_model_8 = search_8.best_estimator_

y_pred_8 = best_model_8.predict(X_test_8)

CLASS_NAMES_8 = [c for c in CLASS_NAMES if c != "seborrheic keratosis"]

print("\nTest Accuracy (8-class):", round(accuracy_score(y_test_8, y_pred_8), 4))
print("\nClassification Report (8-class):")
print(classification_report(y_test_8, y_pred_8, target_names=CLASS_NAMES_8, zero_division=0))

CatBoost

In [ ]:
#STEP 26 — CatBoost (install + train)

!pip install catboost --quiet

from catboost import CatBoostClassifier

#CatBoost handles class imbalance natively — no SMOTE needed
#using original training data not SMOTE'd
cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    random_seed=SEED,
    verbose=50,
    auto_class_weights="Balanced"
)

cat_model.fit(X_train, y_train)

y_pred_cat = cat_model.predict(X_test).flatten()

print("Test Accuracy (CatBoost):", round(accuracy_score(y_test, y_pred_cat), 4))
print("\nClassification Report (CatBoost):")
print(classification_report(y_test, y_pred_cat, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#STEP 27 — CatBoost native cross validation tuning

from catboost import CatBoostClassifier, cv, Pool
import pandas as pd

#create a pool object
train_pool = Pool(X_train, y_train)

#parameters to test
params = {
    "iterations": 500,
    "learning_rate": 0.1,
    "depth": 6,
    "loss_function": "MultiClass",
    "eval_metric": "TotalF1",
    "random_seed": SEED,
    "auto_class_weights": "Balanced",
    "early_stopping_rounds": 50,  #stops if no improvement after 50 rounds
    "verbose": 50
}

#native catboost CV
cv_results = cv(
    pool=train_pool,
    params=params,
    fold_count=3,
    shuffle=True,
    partition_random_seed=SEED,
    plot=False,
    verbose=50
)

print("Best iteration:", cv_results["test-TotalF1-mean"].idxmax())
print("Best CV F1:", round(cv_results["test-TotalF1-mean"].max(), 4))

In [ ]:
#STEP 27 — CatBoost Hyperparameter Tuning

from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostClassifier

param_dist_cat = {
    "iterations":        [200, 300, 400, 500],
    "learning_rate":     [0.05, 0.1, 0.15],
    "depth":             [4, 6, 8],
    "l2_leaf_reg":       [1, 3, 5],
    "border_count":      [32, 64, 128],
}

cat_tuned = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    random_seed=SEED,
    auto_class_weights="Balanced",
    verbose=0
)

search_cat = RandomizedSearchCV(
    estimator=cat_tuned,
    param_distributions=param_dist_cat,
    n_iter=20,
    scoring="f1_macro",
    n_jobs=-1,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    verbose=1,
    random_state=SEED
)

search_cat.fit(X_train, y_train)

print("\nBEST PARAMETERS (CatBoost):")
print(json.dumps(search_cat.best_params_, indent=2))
print("\nBest CV F1 macro:", round(search_cat.best_score_, 4))

best_cat = search_cat.best_estimator_

y_pred_cat_tuned = best_cat.predict(X_test).flatten()

print("\nTest Accuracy (CatBoost tuned):", round(accuracy_score(y_test, y_pred_cat_tuned), 4))
print("\nClassification Report (CatBoost tuned):")
print(classification_report(y_test, y_pred_cat_tuned, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#STEP 27 — CatBoost with built-in grid search

from catboost import CatBoostClassifier, Pool

#try a few key configurations manually and pick the best
configs = [
    {"iterations": 300, "learning_rate": 0.1, "depth": 6},
    {"iterations": 400, "learning_rate": 0.05, "depth": 8},
    {"iterations": 500, "learning_rate": 0.1, "depth": 8},
]

best_f1 = 0
best_cat = None
best_config = None

for config in configs:
    model = CatBoostClassifier(
        **config,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        random_seed=SEED,
        auto_class_weights="Balanced",
        verbose=0
    )
    model.fit(X_train, y_train)
    y_pred_temp = model.predict(X_test).flatten()
    f1 = accuracy_score(y_test, y_pred_temp)
    print(f"Config {config} -> Accuracy: {round(f1, 4)}")
    if f1 > best_f1:
        best_f1 = f1
        best_cat = model
        best_config = config

print("\nBest config:", best_config)
y_pred_cat_tuned = best_cat.predict(X_test).flatten()
print("\nTest Accuracy:", round(accuracy_score(y_test, y_pred_cat_tuned), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_cat_tuned, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#STEP 27 — CatBoost best config (fast)

best_cat = CatBoostClassifier(
    iterations=400,
    learning_rate=0.1,
    depth=6,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    random_seed=SEED,
    auto_class_weights="Balanced",
    verbose=50
)

best_cat.fit(X_train, y_train)

y_pred_cat_tuned = best_cat.predict(X_test).flatten()

print("Test Accuracy (CatBoost):", round(accuracy_score(y_test, y_pred_cat_tuned), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_cat_tuned, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#STEP 28 — CatBoost Confusion Matrix + ROC-AUC

from sklearn.preprocessing import label_binarize

#use the best catboost model (300 iterations from STEP 26)
y_pred_cat_final = cat_model.predict(X_test).flatten()
y_proba_cat = cat_model.predict_proba(X_test)

#confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred_cat_final, labels=list(range(9))),
    display_labels=CLASS_NAMES
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=True
)

ax.set_title("Confusion Matrix — CatBoost (9-class)", fontsize=16, pad=12)
ax.set_xlabel("Predicted label", fontsize=13)
ax.set_ylabel("True label", fontsize=13)

plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=10)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix_catboost.png"), dpi=150)
plt.show()

#ROC-AUC
y_test_bin = label_binarize(y_test, classes=list(range(9)))

auc_macro_cat = roc_auc_score(
    y_test_bin,
    y_proba_cat,
    average="macro",
    multi_class="ovr"
)

print("ROC-AUC macro (CatBoost):", round(auc_macro_cat, 4))

plt.figure(figsize=(10, 7))

for i, cname in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_cat[:, i])
    plt.plot(fpr, tpr, label=cname)

plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.xlabel("False Positive Rate", fontsize=13)
plt.ylabel("True Positive Rate", fontsize=13)
plt.title(f"ROC Curves — CatBoost (9-class) | macro AUC = {auc_macro_cat:.3f}", fontsize=15)
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "roc_curves_catboost.png"), dpi=150)
plt.show()

print("Saved confusion matrix and ROC curves.")